# Step 28 — Re-extract all 4,407 reports with Qwen2.5-14B

**Settings: `GPU T4 x2`, `Internet On`.**
**Inputs:** competition data, `rsna-knee-03-extract-all` (for `gold58.csv`)

## Why

The pilot on 58 gold studies was decisive:

| config | macro AUC | vs current |
|---|---|---|
| 7B greedy (your current labels) | 0.8382 | — |
| 7B self-consistency | 0.8431 | +0.005 |
| **14B greedy** | **0.8712** | **+0.033** |
| 14B self-consistency | 0.8785 | +0.041 |

Model capacity was the bottleneck, not sampling. Self-consistency bought +0.005 on 7B and +0.007 on top of 14B — both marginal. Going 7B to 14B bought **+0.033**. The multilingual reports were simply beyond the smaller model.

**We use greedy, not self-consistency.** The extra +0.008 costs 21 more GPU-hours (31 vs 10), and with 31 days left that time is better spent retraining. The gap is also measured on 58 studies, close to the noise floor.

## Two commits

Measured at 4.1s per generation, both prompts over 4,407 reports is ~10 GPU-hours. Split by prompt:

- **Commit 1:** `PASS = "cat"` — categorical, ~5.1 h
- **Commit 2:** `PASS = "num"` — graded, ~5.1 h

Each writes its own raw output. Cell 6 blends whatever exists and scores against the 58 gold.

Checkpoints every 10 batches, token-budget batching, OOM retry — the same machinery that survived the original extraction.

### 1 · Configure

In [ ]:
PASS  = "cat"          # <-- "cat" for commit 1, then "num" for commit 2
MODEL = "Qwen/Qwen2.5-14B-Instruct"
print(f"pass: {PASS} | model: {MODEL}")

### 2 · Setup

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import re, gc, json, time, glob, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score

ROOT   = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train  = pd.read_csv(f"{ROOT}/train.csv")
gold   = pd.read_csv(glob.glob("/kaggle/input/notebooks/*/*/gold58.csv")[0])
LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()

KEYMAP = {"acl":"ACL","mcl":"MCL","medial_meniscus":"Medial Meniscus",
          "lateral_meniscus":"Lateral Meniscus","medial_oa":"Medial OA",
          "lateral_oa":"Lateral OA","pf_oa":"PF OA","effusion":"Effusion",
          "synovitis":"Synovitis","bakers":"Baker's","contusion":"Contusion",
          "fracture":"Fracture"}
KEYS = list(KEYMAP)

reports = train["Report"].fillna("").astype(str).tolist()
print(f"reports: {len(reports):,} | GPU: {torch.cuda.get_device_name(0)}")

### 3 · Prompts — identical to the 7B run, so only model size changes

In [ ]:
SYS_CAT = """You are a musculoskeletal radiologist extracting structured findings from knee MRI reports.
The report may be in ANY language. Read it in its original language.

For each finding, decide: PRESENT, ABSENT (negated/normal/intact), or NOT_MENTIONED.

- acl: anterior cruciate ligament tear or injury
- mcl: medial collateral ligament tear or injury
- medial_meniscus: medial meniscus tear
- lateral_meniscus: lateral meniscus tear
- medial_oa: MEDIAL tibiofemoral osteoarthritis / chondral loss
- lateral_oa: LATERAL tibiofemoral osteoarthritis / chondral loss
- pf_oa: patellofemoral osteoarthritis / chondromalacia
- effusion: joint effusion
- synovitis: synovial thickening or inflammation
- bakers: Baker's / popliteal cyst
- contusion: bone contusion, bruise, marrow oedema from trauma
- fracture: fracture of any bone

RULES:
1. Negation decides. "ACL intact", "menisci normal", "no tear" => absent.
2. Degenerative signal that is NOT a tear => absent for meniscus/ligament.
3. Distinguish MEDIAL from LATERAL carefully.
4. Never guess: if not referred to, say not_mentioned.

Reply ONLY JSON:
{"acl":"present|absent|not_mentioned","mcl":"...","medial_meniscus":"...","lateral_meniscus":"...","medial_oa":"...","lateral_oa":"...","pf_oa":"...","effusion":"...","synovitis":"...","bakers":"...","contusion":"...","fracture":"..."}"""


SYS_NUM = """You are a musculoskeletal radiologist reading knee MRI reports and estimating how likely each of twelve findings is to be truly present.

The report may be in ANY language. Read it in its original language.

For EACH finding give an integer 0-100: the likelihood an expert reviewing the actual images would mark it present.

  0-10    explicitly negated ("ACL intact", "menisci normal", "no effusion")
  11-30   not mentioned at all
  31-55   equivocal, borderline, trace, minimal, degenerative change short of a tear
  56-80   clearly described but mild, or described indirectly
  81-100  explicitly and unambiguously present

THE TWELVE FINDINGS (match on MEANING, not exact wording):
acl              ACL tear, rupture, discontinuity
mcl              MCL tear, rupture, high-grade sprain
medial_meniscus  medial meniscus TEAR (degeneration alone is NOT a tear -> low)
lateral_meniscus lateral meniscus TEAR (same rule)
medial_oa        MEDIAL compartment: osteoarthritis, chondral loss/thinning, joint space
                 narrowing, subchondral sclerosis, osteophytes
lateral_oa       LATERAL compartment: same features
pf_oa            PATELLOFEMORAL (patella/trochlea): same, chondromalacia patellae,
                 retropatellar cartilage thinning
effusion         joint effusion, intra-articular fluid (trace/small counts, ~50-65)
synovitis        synovial thickening, enhancement, inflammation, plica synovitis
bakers           Baker's cyst, popliteal cyst, gastrocnemio-semimembranosus bursa cyst
contusion        bone contusion, bruise, marrow oedema from trauma
fracture         fracture, cortical break, avulsion, insufficiency/stress fracture

RULES:
- Negation is decisive: "intact", "normal", "unremarkable", "no evidence of" => under 10.
- "Tricompartmental osteoarthritis" raises ALL THREE OA findings.
- Meniscal degeneration without a tear line => 20-40.
- If never referred to, give 15-25. Never exactly 0 for silence.

Reply ONLY JSON of twelve integers:
{"acl":0,"mcl":0,"medial_meniscus":0,"lateral_meniscus":0,"medial_oa":0,"lateral_oa":0,"pf_oa":0,"effusion":0,"synovitis":0,"bakers":0,"contusion":0,"fracture":0}"""

SYSTEM = SYS_CAT if PASS == "cat" else SYS_NUM
print(f"using {'categorical' if PASS == 'cat' else 'graded'} prompt")

### 4 · Load the model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
os.system("pip install -q -U bitsandbytes 2>/dev/null")

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token

t0 = time.time()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                             device_map={"": 0}, dtype=torch.float16)
model.eval()
print(f"loaded in {time.time()-t0:.0f}s | "
      f"{torch.cuda.memory_allocated()/1e9:.1f} GB allocated")

### 5 · Extract

Token-budget batching with OOM retry and checkpointing every 10 batches. Budget is 11,000 tokens — conservative, because 14B in 4-bit leaves far less headroom for KV cache than 7B did.

In [ ]:
CKPT = f"/kaggle/working/ckpt14_{PASS}.json"

def run_extract(reports, system, ckpt, max_new=220, tok_budget=11000):
    out = [None] * len(reports)
    if os.path.exists(ckpt):
        for i, v in json.load(open(ckpt)).items():
            out[int(i)] = v
        print(f"resumed {sum(x is not None for x in out)}/{len(out)}")

    todo = [i for i, x in enumerate(out) if x is None]
    if not todo:
        print("already complete")
        return [x if x is not None else {} for x in out]
    todo.sort(key=lambda i: len(reports[i]))

    SYS_TOK = len(tok(system).input_ids)
    batches, cur, cur_max = [], [], 0
    for i in todo:
        est = min(len(reports[i]) // 3 + SYS_TOK + max_new, 3072)
        if cur and max(cur_max, est) * (len(cur) + 1) > tok_budget:
            batches.append(cur); cur, cur_max = [i], est
        else:
            cur.append(i); cur_max = max(cur_max, est)
    if cur: batches.append(cur)
    print(f"{len(todo)} reports -> {len(batches)} batches")

    def save():
        json.dump({str(i): v for i, v in enumerate(out) if v is not None}, open(ckpt, "w"))

    t0, done0, n, bn = time.time(), len(reports) - len(todo), 0, 0
    while bn < len(batches):
        idx = batches[bn]
        while True:
            try:
                texts = [tok.apply_chat_template(
                            [{"role": "system", "content": system},
                             {"role": "user",   "content": f"REPORT:\n{reports[i][:6000]}"}],
                            tokenize=False, add_generation_prompt=True) for i in idx]
                enc = tok(texts, return_tensors="pt", padding=True,
                          truncation=True, max_length=3072).to(model.device)
                with torch.no_grad():
                    g = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                                       temperature=None, top_p=None, top_k=None,
                                       pad_token_id=tok.pad_token_id)
                new = g[:, enc.input_ids.shape[1]:]
                for j, i in enumerate(idx):
                    txt = tok.decode(new[j], skip_special_tokens=True)
                    m = re.search(r"\{.*?\}", txt, re.S)
                    try:    out[i] = json.loads(m.group(0)) if m else {}
                    except Exception: out[i] = {}
                del enc, g, new
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); gc.collect()
                if len(idx) == 1:
                    out[idx[0]] = {}; break
                half = len(idx) // 2
                print(f"  OOM -> splitting {len(idx)} into {half}+{len(idx)-half}", flush=True)
                batches.insert(bn + 1, idx[half:])
                idx = idx[:half]

        n += len(idx); bn += 1
        if bn % 10 == 0 or bn == len(batches):
            save()
            el = time.time() - t0
            print(f"  {done0+n}/{len(reports)}  {el/max(n,1):.2f}s/rep  "
                  f"eta {(len(todo)-n)*el/max(n,1)/60:.0f} min", flush=True)
    save()
    return [x if x is not None else {} for x in out]


res = run_extract(reports, SYSTEM, CKPT)
fails = sum(1 for x in res if not x)
print(f"\nparse failures: {fails} / {len(res)}")

pd.DataFrame(res).assign(StudyInstanceUID=train.StudyInstanceUID.values) \
  .to_csv(f"/kaggle/working/raw14_{PASS}.csv", index=False)
print(f"wrote raw14_{PASS}.csv")

### 6 · Blend and validate

Runs whenever both passes exist — after commit 2, or after commit 1 if you attach commit 1's output as an input.

Target: the pilot measured **0.871** for 14B greedy on the 58 gold studies, against **0.838** for your current labels.

In [ ]:
CAT2NUM = {"present": 90.0, "not_mentioned": 20.0, "absent": 5.0}
W = {"acl":0.80,"mcl":0.80,"medial_meniscus":0.35,"lateral_meniscus":0.35,
     "medial_oa":0.20,"lateral_oa":0.20,"pf_oa":0.20,"effusion":0.20,
     "synovitis":0.35,"bakers":0.50,"contusion":0.50,"fracture":0.50}

def find(p):
    hits = glob.glob(f"/kaggle/working/{p}") + glob.glob(f"/kaggle/input/notebooks/*/*/{p}")
    return hits[0] if hits else None

f_cat, f_num = find("raw14_cat.csv"), find("raw14_num.csv")
print("cat:", f_cat, "\nnum:", f_num)

if f_cat and f_num:
    dc = pd.read_csv(f_cat).set_index("StudyInstanceUID")
    dn = pd.read_csv(f_num).set_index("StudyInstanceUID")
    ids = train["StudyInstanceUID"].values

    def cnum(v): return CAT2NUM.get(str(v).lower(), 20.0)
    def nnum(v):
        try:    return min(max(float(v), 0), 100)
        except Exception: return 20.0

    blend = pd.DataFrame({
        k: [W[k]*cnum(c) + (1-W[k])*nnum(n)
            for c, n in zip(dc.loc[ids, k].values, dn.loc[ids, k].values)]
        for k in KEYS})
    blend.insert(0, "StudyInstanceUID", ids)
    blend.to_csv("/kaggle/working/pseudo_labels_14b.csv", index=False)
    print("\nwrote pseudo_labels_14b.csv")

    gl = gold["StudyInstanceUID"].tolist()
    P = blend.set_index("StudyInstanceUID").loc[gl, KEYS].values
    Ytr = gold.set_index("StudyInstanceUID").loc[gl, LABELS].values.astype(int)
    OLD = {"ACL":0.881,"MCL":0.880,"Medial Meniscus":0.882,"Lateral Meniscus":0.761,
           "Medial OA":0.943,"Lateral OA":0.857,"PF OA":0.801,"Effusion":0.793,
           "Synovitis":0.729,"Baker's":0.891,"Contusion":0.799,"Fracture":0.844}
    per = []
    print(f"\n{'label':<18}{'7B':>8}{'14B':>8}{'delta':>8}")
    for c, l in enumerate(LABELS):
        y = Ytr[:, c]
        if y.min() == y.max(): continue
        a = roc_auc_score(y, P[:, c]); per.append(a)
        flag = "  <--" if a - OLD[l] > 0.04 else ""
        print(f"{l:<18}{OLD[l]:>8.3f}{a:>8.3f}{a-OLD[l]:>+8.3f}{flag}")
    print(f"\nMACRO  7B 0.838  ->  14B {np.mean(per):.4f}   ({np.mean(per)-0.838:+.4f})")
    print("pilot predicted 0.871")
else:
    print("\nonly one pass done so far - run the other commit, then this cell")